# L08 — 3D reconstruction with structured light

Computational Imaging, 2026. Template notebook: rename it to `L08-solution.ipynb` and submit it in the archive `Team-ID-L08.zip` as described in the handout. Section and step numbers (e.g. `B.2.3`) are those of the handout.

### Important implementation details

1. Functions marked **OWN** must be written by you with NumPy array operations only (no ready-made implementation from OpenCV, scikit-image, SciPy or similar). Everything that is not marked OWN may use any library: loading and saving files, conversion of data types and color, resizing, the fast Fourier transform, plotting, and whatever a step allows explicitly.
1. RGB images are NumPy arrays (`ndarray`) of shape (height, width, 3) with float values in [0, 1]; grayscale images have shape (height, width).
1. You get points for a step only if its result (image, plot, number or written answer) is rendered in the submitted notebook. The comment `Expected output` of every cell says what must be visible.
1. Keep the names and the first arguments of the given functions; you may add arguments with default values and any number of helper functions.
1. Please respect the folder structure (you may add subfolders):
    - `setup-ref` — data of the reference setup (provided): `calibration-data`, `reconstruction-dataset`, `panel_upper_points.npy`, `panel_lower_points.npy`
    - `setup-act` — data of your setup, same structure, plus `structured-light-patterns`, `calibration-dataset` (two photos per pose), `plane-dataset`
    - `setup-***/calibration-data` — eight .npy files: intrinsic matrix, distortion vector, rotation matrix, translation vector of camera and projector (world to device)
1. You may use any library; only the functions marked OWN are restricted.
1. Use an interactive 3D plot (`%matplotlib notebook` or `widget`) while you work, and a static view in the submitted notebook.

### General imports

In [ ]:
%matplotlib inline
import math
import os
from glob import glob

import cv2 as cv
import matplotlib.pyplot as plt  # You may use a different plotting engine
import numpy as np
import itertools

### Auxiliary functions (ready to use)

In [ ]:
def display_images(cont, cols=2, cmap='gray', vmin=None, vmax=None):
    """Displays images from a dictionary {caption: image} in a grid

    Args:
        cont: dictionary with images (height x width x 3, or height x width)
        cols: number of columns of the grid
        cmap, vmin, vmax: passed to imshow for single-channel images
    """

    # This function is ready to use. DO NOT CHANGE IT unless it is really necessary.

    rows = math.ceil(len(cont) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), squeeze=False, constrained_layout=True)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (caption, image) in zip(axes.ravel(), cont.items()):
        ax.set_title(caption)
        if image.ndim == 3:
            ax.imshow(np.clip(image, 0, 1))
        else:
            ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.show()

In [ ]:
def load_images(names, display=True, max_side=1500):
    """Loads image files into a dictionary {caption: RGB image as float array in [0, 1]}

    Args:
        names: list of image files to load (caption = file name without extension)
        display: flag that defines if images should be displayed
        max_side: images with a longer side are downsized to this size (None: keep full resolution)

    Returns:
        Dictionary with loaded images
    """

    # This function is ready to use. DO NOT CHANGE IT unless it is really necessary.

    images = {}
    for name in sorted(names):
        image = cv.imread(name, cv.IMREAD_COLOR)
        if image is None:
            print('Cannot read', name)
            continue
        image = cv.cvtColor(image, cv.COLOR_BGR2RGB).astype(np.float64) / 255.0
        if max_side is not None and max(image.shape[:2]) > max_side:
            scale = max_side / max(image.shape[:2])
            image = cv.resize(image, None, fx=scale, fy=scale, interpolation=cv.INTER_AREA)
        images[os.path.splitext(os.path.basename(name))[0]] = image
    if not images:
        print('No images loaded')
    elif display:
        display_images(images)
    return images

In [ ]:
def to_gray(image):
    """Converts an RGB image (float, linear weights of Rec. 709) to a single-channel image"""

    # This function is ready to use.

    return image @ np.array([0.2126, 0.7152, 0.0722]) if image.ndim == 3 else image

In [ ]:
def load_calibration(path):
    """Loads calibration data for camera and projector from files stored in specified folder

    Args:
        path: path to folder where calibration files are stored

    Returns:
        a tuple of 2 lists with camera and projector calibration data (in order):
            - intrinsic matrix
            - distortion vector
            - rotation matrix
            - translation vector
        Rotation matrix and translation vector are used for transition from world into camera/projector coordinates
    """

    # This function is ready to use. DO NOT CHANGE IT UNLESS IT IS REALLY NECESSARY.

    names = ['-'.join(x) for x in itertools.product(['camera', 'projector'],
                                                    ['intrinsic-matrix.npy',
                                                     'distortion-vector.npy',
                                                     'rotation-matrix.npy',
                                                     'translation-vector.npy'])]

    return ([np.load(os.path.join(path, x)) for x in names[:4]],
            [np.load(os.path.join(path, x)) for x in names[4:]])

In [ ]:
def save_calibration(path, camera, projector):
    """Saves calibration data in the format that load_calibration() reads

    Args:
        path: folder (created if necessary)
        camera, projector: lists [intrinsic matrix, distortion vector, rotation matrix, translation vector]
    """

    # This function is ready to use.

    os.makedirs(path, exist_ok=True)
    keys = ['intrinsic-matrix', 'distortion-vector', 'rotation-matrix', 'translation-vector']
    for device, data in (('camera', camera), ('projector', projector)):
        for key, value in zip(keys, data):
            np.save(os.path.join(path, f'{device}-{key}.npy'), np.asarray(value))

In [ ]:
def load_projector_panel_points(path_upper, path_lower):
    """Loads coordinates of projector panel points that will be used during 3D reconstruction

    Args:
        path_upper: path to file that contains upper projector panel coordinates
        path_lower: path to file that contains lower projector panel coordinates

    Returns:
        Array of 2D points (2xMx2). Index 0 of the first dimension: upper points, index 1: lower points.
    """

    # This function is ready to use. DO NOT CHANGE IT UNLESS IT IS REALLY NECESSARY.
    # "Upper" and "lower" are defined in the world coordinate system. On the projector panel they are inversed.

    u_px = np.load(path_upper)
    l_px = np.load(path_lower)

    return np.rollaxis(np.dstack((u_px, l_px)), 2, 0)

## P  Pre-lab derivation: triangulation with a plane of light  [3 pts]

*`TODO (P.1–P.3):`* Insert the scan or photos of your handwritten derivation (at most two A4 pages), e.g. `![page 1](data/prelab-1.jpg)`.

In [ ]:
# TODO (P.3.3): Repeat your numerical prediction with the values of your own setup: your z, b, f in pixels; predicted depth error for an edge localization error of 0.5 pixel
# Expected output: the input values and the predicted numbers, printed with units

## A  Data acquisition  [5 pts]

### A.1  Assembly  [max. 1 pt]

In [ ]:
# TODO (A.1.3): Display the photo of the finished setup with your student IDs; print baseline and object distance
# Expected output: 1 image, 2 numbers
print("baseline : 17 cm")
print("object distance : 54 cm")

### A.2  Calibration dataset  [max. 1 pt]

In [ ]:
# TODO (A.2): Display four poses (white-frame photo and pattern photo of each); store the indexes of these four poses for later steps
# Expected output: 4 image pairs

### A.3  Scanning patterns  [max. 1 pt]

In [ ]:
def generate_slset(n, shape=(800, 1280), path='setup-act/structured-light-patterns'):
    """Generates the scanning patterns (one dark-light edge per pattern) and the panel coordinates of the upper and lower end of every edge

    Args:
        n: number of patterns (at least 20)
        shape: resolution of the projector panel
        path: folder where the patterns, a white and a black frame are saved

    Returns:
        Array (2, n, 2) of panel points, as returned by load_projector_panel_points()
    """

    # TODO (A.3.1): Write the function
    # OWN: NumPy array operations only, no ready-made implementation from a library
    # Notes:
    #   - generate at the native resolution of the projector to avoid interpolation
    #   - save the points as panel_upper_points.npy and panel_lower_points.npy

    return np.zeros((2, n, 2))

In [ ]:
# TODO (A.3.2): Display four patterns with their panel points
# Expected output: 4 images

### A.4–A.5  Scans of the object and of the plane  [max. 2 pts]

In [ ]:
# TODO (A.4–A.5): Display four photos of each scan, and the white and black photos; print the two caliper measurements
# Expected output: 2 rows of 4 images + 4 images, 2 numbers

## B  Core processing  [8 pts]

### B.1  Reconstruction with the reference setup  [max. 3 pts]

In [ ]:
# TODO (B.1.1): Load images, calibration data (load_calibration) and panel points (load_projector_panel_points) of the reference setup; select 4 images for display
# Expected output: 4 images, shapes of the loaded arrays

*`TODO (B.1.2):`* Describe the purpose of each component of the calibration data.

In [ ]:
def detect_edge(image, white, black, n=20):
    """Sub-pixel positions of the dark-light edge in a scan photo

    Args:
        image, white, black: grayscale photos under the pattern, the white and the black frame
        n: minimal number of points

    Returns:
        Array (N, 2) of sensor coordinates (x, y)
    """

    # TODO (B.1.3): Write the function
    # OWN: NumPy array operations only, no ready-made implementation from a library
    # Notes:
    #   - in every row (or column): where does the intensity cross (white + black) / 2? interpolate linearly
    #   - skip rows without sufficient contrast between white and black

    return np.zeros((n, 2))

In [ ]:
# TODO (B.1.3): Draw the detected points on the four images
# Expected output: 4 images

In [ ]:
def pixel_to_ray(K, dist, R, t, points):
    """Rays of pixels in world coordinates (P.1)

    Args:
        K, dist, R, t: calibration data of the device
        points: array (N, 2) of pixel coordinates

    Returns:
        (center C, array (N, 3) of directions d)
    """

    # TODO (B.1.4): Write the function
    # OWN: NumPy array operations only, no ready-made implementation from a library
    # Notes:
    #   - undistort the points first (cv2.undistortPoints is allowed)
    #   - C = -R^T t, d = R^T K^-1 x

    return (np.zeros(3), np.zeros((len(points), 3)))

In [ ]:
def intersect(C, d, plane_point, plane_normal):
    """Intersection of rays with a plane (P.2)

    Args:
        C: ray origin
        d: array (N, 3) of ray directions
        plane_point, plane_normal: the plane

    Returns:
        Array (N, 3) of intersection points
    """

    # TODO (B.1.5): Write the function
    # OWN: NumPy array operations only, no ready-made implementation from a library

    return np.zeros((len(d), 3))

In [ ]:
def reconstruct(sensor_points, panel_points, camera, projector):
    """Point cloud from edge points and panel points

    Args:
        sensor_points: list of M arrays (N, 2)
        panel_points: array (2, M, 2)
        camera, projector: calibration data lists

    Returns:
        Array of 3D points
    """

    # TODO (B.1.5): Combine pixel_to_ray() and intersect()

    return np.zeros((400, 3))

In [ ]:
# TODO (B.1.5): Reconstruct all points of all images; display the point cloud from two viewpoints
# Expected output: 2 views, number of points (at least 400)

### B.2  Calibration of the camera  [max. 2 pts]

In [ ]:
# TODO (B.2.1): Detect the corners of the printed checkerboard in the white-frame photos; draw them on the four selected photos
# Expected output: 4 images

In [ ]:
# TODO (B.2.2): World coordinates of the detected corners (three-dot marker = origin, plate = plane z = 0, cell size 10 mm), in the order of the detected corners
# Expected output: shapes of the point arrays

In [ ]:
# TODO (B.2.3): Calibrate the camera; store the pose of the plate for every photo
# Expected output: intrinsic matrix, distortion coefficients, reprojection error

*`TODO (B.2.4):`* Is the focal length in pixels consistent with a 50 mm lens and the pixel pitch? Is the principal point near the center?

### B.3  Calibration of the projector and of the pair  [max. 2 pts]

In [ ]:
# TODO (B.3.1): Detect the features of the projected pattern in the second photo of every pose; draw them on the four selected photos
# Expected output: 4 images

In [ ]:
# TODO (B.3.2): World coordinates of the features: intersect their camera rays with the plane of the plate (pixel_to_ray, intersect)
# Expected output: shapes; plate residuals

In [ ]:
# TODO (B.3.3): Calibrate the projector as an inverse camera
# Expected output: intrinsic matrix, reprojection error

In [ ]:
# TODO (B.3.4): Pose of the projector relative to the camera, averaged over the poses; save everything with save_calibration('setup-act/calibration-data', ...)
# Expected output: rotation, translation, baseline length

### B.4  Reconstruction with your setup  [max. 1 pt]

In [ ]:
# TODO (B.4.1–B.4.2): Run the pipeline on your scan of the object and of the flat board
# Expected output: 2 point clouds, 2 views each

## C  Analysis: prediction vs. measurement  [4 pts]

### C.1  Is the calibration plausible?  [max. 1 pt]

In [ ]:
# TODO (C.1.1–C.1.2): Baseline from the calibration against the tape measurement; two dimensions of the object from the point cloud against the caliper
# Expected output: table with 3 rows

### C.2  Depth noise  [max. 2 pts]

In [ ]:
def fit_plane(points):
    """Least-squares plane through a point cloud

    Args:
        points: array (N, 3)

    Returns:
        (point on the plane, unit normal, RMS distance)
    """

    # TODO (C.2.1): Write the function (SVD)
    # OWN: NumPy array operations only, no ready-made implementation from a library

    return (np.zeros(3), np.array([0, 0, 1.0]), 0.0)

In [ ]:
# TODO (C.2.1): RMS distance of the board points from the fitted plane; residuals as color-coded map
# Expected output: 1 number, 1 map

In [ ]:
# TODO (C.2.2): Edge localization error: RMS residual of the detected edge points of one image from a smooth curve
# Expected output: 1 number in pixels

In [ ]:
# TODO (C.2.3): Predicted depth error (P.3) for this localization error and your z, b against the RMS of the plane fit
# Expected output: table

*`TODO (C.2.4):`* Is the residual map random or does it show a pattern? Attribute systematic features to a component of the calibration or of the edge detection.

### C.3  Limits of the method  [max. 1 pt]

In [ ]:
# TODO (C.3.1): Show the regions of the object that are missing in the point cloud
# Expected output: 1–2 views

*`TODO (C.3.1):`* Explain each with the geometry of camera, projector and object.

*`TODO (C.3.2):`* How would you change the setup to halve the depth noise, and what would you lose?

## +  Bonus  [up to 2 pts]

In [ ]:
# TODO (+.1): Camera, projector (with frusta), plate poses and point cloud in one 3D plot
# Expected output: 1 plot

In [ ]:
# TODO (+.2): Gray-code patterns: dense reconstruction; why Gray code rather than binary code (markdown)
# Expected output: dense point cloud

## Checklist

Before you submit, go through the checklist on the last page of the handout: data complete, pre-lab scan inserted, every step of Parts B and C rendered, archive named as required.